# Crime and Public Safety — When the Coverage Is the Variable

Every source in this corpus so far measures something and tells you, with varying honesty,
how well it measured it. This one measures whatever agencies volunteered to report, and the
set of volunteers changes from year to year and from state to state for reasons that have
nothing to do with crime.

That is not a criticism of the publisher. NIBRS participation is voluntary by statute, the
FBI publishes the coverage, and it normalises its rates by the covered population — all three
of which are more than most publishers do. This notebook verifies each of them rather than
assuming, and then shows why all three together are still not enough.

The short version: **a coverage figure tells you a problem exists, and it does not tell you
the sign of the problem.** Across ten states the correlation between how far coverage moved
and how far the published rate moved is weak, and it contains a sign reversal — most states
that lost coverage saw their rate fall, while Pennsylvania's rose by a third. Neither
movement is about crime. And in the worst case a state reads near-total coverage over a
window in which nine tenths of its monthly offence counts are near zero.

## Learning Objectives

1. **Read a participation series as data**, not as metadata, and notice when the publisher
   has filed it somewhere that invites you not to.
2. **Verify which denominator a published rate uses** rather than inferring it from the name.
3. **Distinguish an outage from an absence**, when the upstream returns a failure that a
   careless client turns into a zero.
4. **Recognise a coverage figure that is true and useless**, and say precisely what it does
   and does not certify.
5. **Refuse a correction you cannot sign**: know why a coverage-weighted adjustment is not
   available here even though the coverage number is.

## Prerequisites

- Handbook §1.4 (QCEW — suppression with a flag) for the contrast. There, what is missing is
  marked. Here, what is missing is an agency that did not volunteer, and nothing marks it.
- Handbook §8.2 (coverage is two numbers, and they support different sentences). This is that
  section's hardest case.

## Data Sources

| endpoint | what a row is |
|---|---|
| `api.usa.gov/crime/fbi/cde/agency/byStateAbbr/{ST}` | one law-enforcement agency, with `is_nibrs` and `nibrs_start_date` |
| `api.usa.gov/crime/fbi/cde/summarized/state/{ST}/{offense}` | one month, with offences, rate, population, **participated population** and coverage |

Periods are `MM-YYYY`; a bare year is rejected with HTTP 400. Access is through an
api.data.gov key, which authenticates across every participating agency.

**On transport.** The FBI backend behind api.usa.gov returns HTTP 503 intermittently and
recovers within seconds — during this build the same endpoint failed and then answered on the
next call, while a different endpoint that had worked fifty-one times in a row failed. The
client retries and, when it gives up, raises an error that names the outage. It never returns
an empty frame, because an empty frame here reads as a place with no crime.

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    CDEUpstreamUnavailable,
    FBI_CDE_COVERAGE_PATH,
    FBI_CDE_REGIME,
    fetch_fbi_cde_agencies,
    fetch_fbi_cde_summarized,
)

OFFENSE = "violent-crime"
FROM_MONTH, TO_MONTH = "01-2016", "12-2023"
BASELINE_YEAR, TRANSITION_YEAR = "2019", "2021"

#: Ten states spanning the participation range, named rather than sampled. The
#: national picture is not computed here: it would need 51 live calls against a
#: flapping upstream, which is not a reproducible artifact.
STATES = {
    "PA": "Pennsylvania", "FL": "Florida", "NY": "New York", "CA": "California",
    "IL": "Illinois", "NJ": "New Jersey", "TX": "Texas", "VA": "Virginia",
    "HI": "Hawaii", "OH": "Ohio",
}
#: The four whose agency rosters are pulled, chosen to span the range.
AGENCY_STATES = ["PA", "FL", "VA", "CA"]

pd.set_option("display.width", 116)

print(f"offense window : {OFFENSE}, {FROM_MONTH} to {TO_MONTH}")
print(f"states         : {len(STATES)}")
print(f"coverage series lives at: {' -> '.join(FBI_CDE_COVERAGE_PATH)}")
print()
print(f"regime: {FBI_CDE_REGIME}")

offense window : violent-crime, 01-2016 to 12-2023
states         : 10
coverage series lives at: tooltips -> Percent of Population Coverage

regime: voluntary NIBRS reporting: the population covered by reporting agencies is a time series that moves for administrative reasons, varies by state, and is not a random sample of places. Coverage is published and rates are normalised by it, and neither is sufficient -- the sign of the residual bias is not recoverable from the coverage figure


In [2]:
"""Retrieve the monthly series and the agency rosters."""
series = {}
unavailable = []
for abbrev in STATES:
    try:
        frame, _ = fetch_fbi_cde_summarized(
            state=abbrev, offense=OFFENSE, from_month=FROM_MONTH, to_month=TO_MONTH
        )
        series[abbrev] = frame
    except CDEUpstreamUnavailable as outage:
        unavailable.append(abbrev)
        print(f"  {abbrev}: upstream unavailable -- {str(outage)[:70]}")

rosters = {}
for abbrev in AGENCY_STATES:
    try:
        frame, _ = fetch_fbi_cde_agencies(state=abbrev)
        rosters[abbrev] = frame
    except CDEUpstreamUnavailable:
        unavailable.append(f"{abbrev} agencies")

print(f"monthly series retrieved : {len(series)} of {len(STATES)} states")
print(f"agency rosters retrieved : {len(rosters)} of {len(AGENCY_STATES)} states")
print(f"unavailable this run     : {unavailable if unavailable else 'none'}")
print()
print("note that the line above records an OUTAGE, not an absence. A state missing")
print("here has not been shown to have no crime data; the gateway did not answer.")

monthly series retrieved : 10 of 10 states
agency rosters retrieved : 4 of 4 states
unavailable this run     : none

note that the line above records an OUTAGE, not an absence. A state missing
here has not been shown to have no crime data; the gateway did not answer.


## 1. The coverage series, and where it is filed

Start with the agency rosters, because that is where the variable originates. Each agency
carries `is_nibrs` and, if it reports, the date it started. Nothing else in this corpus
publishes the date a unit entered the measurement.

In [3]:
"""Who reports, and since when."""
roster_rows = []
for abbrev, frame in rosters.items():
    reporting = int(frame["is_nibrs"].fillna(False).sum())
    roster_rows.append({
        "state": abbrev,
        "agencies": len(frame),
        "reporting NIBRS": reporting,
        "share reporting": reporting / len(frame),
    })
roster_summary = pd.DataFrame(roster_rows).set_index("state").sort_values("share reporting")
print(roster_summary.to_string(formatters={
    "agencies": "{:,.0f}".format, "reporting NIBRS": "{:,.0f}".format,
    "share reporting": "{:.1%}".format}))

joins = pd.concat(
    [f.assign(state=a) for a, f in rosters.items()], ignore_index=True
)
joins = joins[joins["is_nibrs"].fillna(False) & joins["nibrs_start_year"].notna()]
by_year = joins["nibrs_start_year"].astype(int).value_counts().sort_index()
recent = by_year[by_year.index >= 2016]
total_agencies = sum(len(f) for f in rosters.values())

print()
print(f"across these {len(rosters)} states, {total_agencies:,} agencies; "
      f"{int(joins.shape[0]):,} report NIBRS with a recorded start date")
print("agencies joining NIBRS by year:")
running = int(by_year[by_year.index < 2016].sum())
for year, count in recent.items():
    running += int(count)
    print(f"  {year}  joined {int(count):>5,}   cumulative {running:>6,}  "
          f"= {running / total_agencies:>6.1%} of all agencies")
peak = int(recent.idxmax())
print()
print(f"largest single adoption year in this window: {peak} "
      f"({int(recent.max()):,} agencies)")

      agencies reporting NIBRS share reporting
state                                         
PA       1,439             245           17.0%
FL         834             299           35.9%
CA         867             768           88.6%
VA         412             412          100.0%

across these 4 states, 3,552 agencies; 1,724 report NIBRS with a recorded start date
agencies joining NIBRS by year:
  2016  joined     1   cumulative    426  =  12.0% of all agencies
  2018  joined     2   cumulative    428  =  12.0% of all agencies
  2020  joined     8   cumulative    436  =  12.3% of all agencies
  2021  joined   515   cumulative    951  =  26.8% of all agencies
  2022  joined   289   cumulative  1,240  =  34.9% of all agencies
  2023  joined   136   cumulative  1,376  =  38.7% of all agencies
  2024  joined    96   cumulative  1,472  =  41.4% of all agencies
  2025  joined   142   cumulative  1,614  =  45.4% of all agencies
  2026  joined   110   cumulative  1,724  =  48.5% of all agenci

The adoption is not spread evenly across the window and it is not spread evenly across
states. Both of those matter, and the second matters more: a national series assembled from
these agencies is a different set of places each year, weighted towards whichever states
happened to be ready.

Now the part that is easy to miss. The coverage figure **is** published — it sits in the
monthly response under `tooltips` → `Percent of Population Coverage`. A key called `tooltips`
is where a chart library keeps its hover text. The variable that decides whether a crime
series means anything is filed as chart furniture, which is why this client lifts it into a
column and the constant naming that path is pinned in the module.

## 2. Which denominator does the published rate use?

A rate needs a population. With coverage moving between 40% and 100%, the choice between
resident population and *covered* population changes the answer by more than the phenomenon
does. The publisher does not say in the payload. So measure it: reconstruct the rate both
ways and see which reproduces the published figure.

In [4]:
"""Reconstruct the published rate against both candidate denominators."""
checks = []
for abbrev, frame in series.items():
    usable = frame.dropna(subset=["offenses", "published_rate", "population",
                                  "participated_population"])
    usable = usable[usable["participated_population"] > 0]
    if usable.empty:
        continue
    from_total = 100_000 * usable["offenses"] / usable["population"]
    from_covered = 100_000 * usable["offenses"] / usable["participated_population"]
    checks.append({
        "state": abbrev,
        "months": len(usable),
        "err vs total population": (from_total - usable["published_rate"]).abs().mean(),
        "err vs covered population": (from_covered - usable["published_rate"]).abs().mean(),
        "min coverage %": usable["coverage_pct"].min(),
    })

denominator = pd.DataFrame(checks).set_index("state").sort_values("min coverage %")
print("mean absolute error of the reconstructed rate, by denominator")
print(denominator.to_string(formatters={
    "months": "{:,.0f}".format, "err vs total population": "{:.4f}".format,
    "err vs covered population": "{:.4f}".format, "min coverage %": "{:.1f}".format}))

total_err = denominator["err vs total population"].mean()
covered_err = denominator["err vs covered population"].mean()
print()
print(f"mean across states, total-population denominator   : {total_err:.4f}")
print(f"mean across states, covered-population denominator : {covered_err:.4f}")
print(f"the publisher divides by: "
      f"{'COVERED population' if covered_err < total_err else 'TOTAL population'}")
print()
print("states with near-total coverage cannot distinguish the two; the answer comes")
print("from the states where coverage actually falls.")

mean absolute error of the reconstructed rate, by denominator
      months err vs total population err vs covered population min coverage %
state                                                                        
CA        96                  2.9036                    0.0024           20.6
IL        96                  3.5218                    0.0023           35.4
PA        96                  4.0987                    0.0024           39.4
FL        96                  2.1543                    0.0020           42.5
NJ        96                  0.6727                    0.0027           49.8
NY        96                  0.5987                    0.0023           64.5
OH        96                  2.1801                    0.0024           89.6
TX        96                  0.3894                    0.0024           97.4
VA        96                  0.0122                    0.0028           99.8
HI        96                  0.0026                    0.0026          100.0

m

The publisher is doing the right thing, and doing it invisibly. The rate is normalised by the
population the reporting agencies actually serve, so a state that loses half its agencies
does not mechanically halve its crime rate. Any analysis that recomputes a rate from
`offenses` and resident population — the obvious move, since `population` is right there in
the response — will disagree with the published figure by more than the entire year-on-year
variation in crime.

**[rule]** When a response carries two plausible denominators, reconstruct the published
figure before using either. The name of a column is not evidence of what was divided by what.

## 3. A coverage figure can be true and useless

Now the case that should change how you read this source. Coverage is a population statistic:
it measures the population served by agencies in *some* reporting arrangement. It is not a
statement that those agencies reported *this* series, for *these* months, with anything in
them.

In [5]:
"""Where the offence counts and the coverage figure disagree about reality."""
worst_case = None
rows = []
for abbrev, frame in series.items():
    early = frame[frame["month"].str.endswith(("2016", "2017", "2018", "2019", "2020"))]
    if early.empty:
        continue
    near_zero = int((early["offenses"] < 200).sum())
    rows.append({
        "state": abbrev,
        "months 2016-2020": len(early),
        "months under 200 offences": near_zero,
        "share": near_zero / len(early),
        "median monthly offences": early["offenses"].median(),
        "mean coverage %": early["coverage_pct"].mean(),
    })

quality = pd.DataFrame(rows).set_index("state").sort_values("share", ascending=False)
print(f"the {FROM_MONTH[-4:]}-2020 window, before the transition year")
print(quality.to_string(formatters={
    "months 2016-2020": "{:,.0f}".format, "months under 200 offences": "{:,.0f}".format,
    "share": "{:.1%}".format, "median monthly offences": "{:,.0f}".format,
    "mean coverage %": "{:.2f}".format}))

worst = quality.index[0]
print()
print(f"worst case in this set: {worst} ({STATES[worst]})")
print(f"  {int(quality.loc[worst, 'months under 200 offences'])} of "
      f"{int(quality.loc[worst, 'months 2016-2020'])} months under 200 offences")
print(f"  median monthly offences over the window: "
      f"{quality.loc[worst, 'median monthly offences']:,.0f}")
print(f"  mean reported population coverage       : "
      f"{quality.loc[worst, 'mean coverage %']:.2f}%")

the 2016-2020 window, before the transition year
      months 2016-2020 months under 200 offences share median monthly offences mean coverage %
state                                                                                         
FL                  60                        54 90.0%                       4           99.82
PA                  60                         0  0.0%                   3,215           94.93
NY                  60                         0  0.0%                   5,874           98.68
CA                  60                         0  0.0%                  14,698           99.87
IL                  60                         0  0.0%                   4,497           95.55
NJ                  60                         0  0.0%                   1,626          100.00
TX                  60                         0  0.0%                  10,202           99.18
VA                  60                         0  0.0%                   1,523           99.95
H

In [6]:
"""The same state, year by year, so the artifact is unmistakable."""
frame = series[worst]
annual = frame.assign(year=frame["month"].str[-4:]).groupby("year").agg(
    median_offences=("offenses", "median"),
    min_offences=("offenses", "min"),
    max_offences=("offenses", "max"),
    mean_coverage=("coverage_pct", "mean"),
)
print(f"{STATES[worst]}, monthly violent-crime offences by year")
print(annual.to_string(formatters={
    "median_offences": "{:,.0f}".format, "min_offences": "{:,.0f}".format,
    "max_offences": "{:,.0f}".format, "mean_coverage": "{:.2f}".format}))
print()
print("a series whose median month is near zero and whose maximum month is in the")
print("tens of thousands is not a measurement of monthly crime. The coverage column")
print("does not fall when that happens, because nothing about it is measuring this.")

Florida, monthly violent-crime offences by year
     median_offences min_offences max_offences mean_coverage
year                                                        
2016               0            0       75,308         99.78
2017               6            1       85,497         99.94
2018               2            0       81,896         99.91
2019               7            2       81,025         99.76
2020               6            0       82,936         99.69
2021           1,742        1,443        2,012         46.91
2022           2,888        2,419       30,066         84.34
2023           3,344        2,763       16,251         82.97

a series whose median month is near zero and whose maximum month is in the
tens of thousands is not a measurement of monthly crime. The coverage column
does not fall when that happens, because nothing about it is measuring this.


This is the finding to carry away. An analyst doing the careful thing — checking the coverage
field before using the series — is waved through. Coverage answers "what share of people live
in places served by a reporting agency". It does not answer "did those agencies submit usable
counts for these months", and the two come apart badly.

**[rule]** A coverage statistic certifies the denominator, never the numerator. Before
trusting a series, look at the counts themselves for the period you intend to use: a median
month near zero beside a maximum month in the tens of thousands is a reporting artifact,
whatever the coverage column says.

## 4. Why you cannot correct for it

The tempting next move is a coverage-weighted adjustment: scale the rate, or reweight states,
so the transition years become comparable. That requires knowing which direction the
non-reporting agencies would have moved the number. Check whether the data support that.

In [7]:
"""Does the change in coverage predict the change in the published rate?"""
movements = []
for abbrev, frame in series.items():
    base = frame[frame["month"].str.endswith(BASELINE_YEAR)]
    shift = frame[frame["month"].str.endswith(TRANSITION_YEAR)]
    if base.empty or shift.empty:
        continue
    rate_base, rate_shift = base["published_rate"].mean(), shift["published_rate"].mean()
    cov_base, cov_shift = base["coverage_pct"].mean(), shift["coverage_pct"].mean()
    movements.append({
        "state": abbrev,
        f"rate {BASELINE_YEAR}": rate_base,
        f"rate {TRANSITION_YEAR}": rate_shift,
        "rate change %": 100 * (rate_shift - rate_base) / rate_base,
        "coverage change pp": cov_shift - cov_base,
    })

moves = pd.DataFrame(movements).set_index("state").sort_values("coverage change pp")
print(f"{BASELINE_YEAR} to {TRANSITION_YEAR}, annual means")
print(moves.to_string(formatters={
    f"rate {BASELINE_YEAR}": "{:.2f}".format, f"rate {TRANSITION_YEAR}": "{:.2f}".format,
    "rate change %": "{:+.1f}".format, "coverage change pp": "{:+.1f}".format}))

pearson, p_pearson = stats.pearsonr(moves["coverage change pp"], moves["rate change %"])
spearman, p_spearman = stats.spearmanr(moves["coverage change pp"], moves["rate change %"])
print()
print(f"correlation between coverage change and rate change, n={len(moves)} states")
print(f"  Pearson  r   = {pearson:+.3f}  (p={p_pearson:.3f})")
print(f"  Spearman rho = {spearman:+.3f}  (p={p_spearman:.3f})")
# Section 3 disqualified some of these series. A state whose monthly counts are
# not a measurement cannot contribute a meaningful rate change, and leaving it in
# would let this notebook use a number it has just finished disqualifying.
UNUSABLE_SHARE = 0.5
disqualified = sorted(quality.index[quality["share"] > UNUSABLE_SHARE])
screened = moves.drop(index=[st for st in disqualified if st in moves.index])
print()
print(f"disqualified by the section 3 screen (over {UNUSABLE_SHARE:.0%} of 2016-2020 "
      f"months under 200 offences): {disqualified}")
if len(screened) < len(moves):
    ps, pps = stats.pearsonr(screened["coverage change pp"], screened["rate change %"])
    ss, pss = stats.spearmanr(screened["coverage change pp"], screened["rate change %"])
    print(f"  recomputed on the remaining {len(screened)} states:")
    print(f"    Pearson  r   = {ps:+.3f}  (p={pps:.3f})")
    print(f"    Spearman rho = {ss:+.3f}  (p={pss:.3f})")
else:
    ps, ss = pearson, spearman
    print("  nothing was disqualified; the figures above stand")

losers = screened[screened["coverage change pp"] < -20]
rose = int((losers["rate change %"] > 0).sum())
fell = int((losers["rate change %"] < 0).sum())
print()
print(f"among the {len(losers)} screened states losing more than 20 points of coverage,")
print(f"  the rate fell in {fell} and rose in {rose}")
print(f"  largest fall: {losers['rate change %'].min():+.1f}% "
      f"({losers['rate change %'].idxmin()})")
print(f"  largest rise: {losers['rate change %'].max():+.1f}% "
      f"({losers['rate change %'].idxmax()})")

2019 to 2021, annual means
      rate 2019 rate 2021 rate change % coverage change pp
state                                                     
CA        36.72     29.40         -19.9              -73.3
PA        24.82     33.62         +35.5              -52.9
FL        31.49     16.98         -46.1              -52.8
IL        35.22     24.81         -29.5              -42.2
NY        30.28      6.47         -78.6              -29.5
NJ        17.28     15.20         -12.0              -24.6
TX        35.58     38.35          +7.8               -1.0
OH        26.39     28.40          +7.6               -0.5
VA        17.73     18.97          +7.0               -0.0
HI        22.51     22.49          -0.1               +0.0

correlation between coverage change and rate change, n=10 states
  Pearson  r   = +0.293  (p=0.411)
  Spearman rho = +0.297  (p=0.405)

disqualified by the section 3 screen (over 50% of 2016-2020 months under 200 offences): ['FL']
  recomputed on the remaining 9 s

Be careful what that licenses, because there is a tendency in it and a tendency is not a rule.

Among the screened states that lost real coverage the published rate mostly **fell**, which is
what happens when the agencies that dropped out served higher-crime jurisdictions than the
state average: removing them takes proportionally more out of the numerator than out of the
denominator. That is a direction, and in this set it is the common one.

It is still not a correction, for two reasons visible in the output. There is a sign reversal
— Pennsylvania's rate rose by a third while its coverage more than halved — so a coefficient
fitted here would push that state the wrong way. And the correlation is weak, nowhere near
conventional significance, on ten states that were **named rather than sampled**, so it
carries no standard error and nothing to generalise with.

The mechanism is not mysterious. Whether losing an agency raises or lowers the measured rate
depends on whether that agency's jurisdiction was above or below the state's average, and
agencies did not leave the measurement at random — they entered it when they were
administratively ready, which correlates with size, budget and urbanisation in ways that
differ by state. A single coverage number cannot encode that.

**[rule]** A coverage figure supports refusing a comparison. It does not support repairing
one. If the analysis needs the transition years, the honest move is agency-level: hold the
set of reporting agencies fixed and compare only those, accepting a smaller, named universe
rather than a national number that silently changes composition.

## 5. An outage is not a zero

One last hazard, and it belongs in a notebook rather than a footnote because the failure is
silent and the wrong handling is the natural one.

In [8]:
"""What the client does when the gateway will not answer."""
class _AlwaysUnavailable:
    """Stands in for the gateway during an outage. No network is used here."""

    def __init__(self) -> None:
        self.attempts = 0

    def get(self, *args, **kwargs):
        self.attempts += 1
        class _Refused:
            status_code = 503
            content = b"upstream connect error or disconnect/reset before headers"
            headers = {"content-type": "text/plain"}
            text = content.decode()
        return _Refused()

gateway = _AlwaysUnavailable()
try:
    fetch_fbi_cde_summarized(
        state="PA", offense=OFFENSE, from_month="01-2021", to_month="12-2021",
        session=gateway,
    )
except CDEUpstreamUnavailable as outage:
    print(f"attempts made before giving up: {gateway.attempts}")
    print(f"error type raised             : {type(outage).__name__}")
    print()
    print("message:")
    for line in str(outage).split(". "):
        if line.strip():
            print(f"  {line.strip()}.")

attempts made before giving up: 5
error type raised             : CDEUpstreamUnavailable

message:
  CDE returned HTTP 503 on all 5 attempts for /summarized/state/PA/violent-crime.
  The FBI backend behind api.usa.gov flaps and recovers within seconds, so this says nothing about whether the data exists.
  Retry later; do NOT record it as an absence..


The alternative — returning an empty frame — produces a table in which Pennsylvania has no
violent crime in 2021. It would pass every structural check in this corpus: the schema is
right, the types are right, nothing is null, and the number is a number.

## Findings

In [9]:
"""FINDINGS"""
findings = []

findings.append((
    "F1",
    f"NIBRS participation is a variable, not a constant. Across {len(rosters)} states and "
    f"{total_agencies:,} agencies, the share reporting ranges from "
    f"{roster_summary['share reporting'].min():.1%} ({roster_summary.index[0]}) to "
    f"{roster_summary['share reporting'].max():.1%} ({roster_summary.index[-1]}). "
    f"The largest single adoption year in {FROM_MONTH[-4:]}-{TO_MONTH[-4:]} was {peak}, "
    f"with {int(recent.max()):,} agencies joining."
))
findings.append((
    "F2",
    f"The coverage series is published, and it is filed under "
    f"'{' -> '.join(FBI_CDE_COVERAGE_PATH)}' -- a key a chart library uses for hover text. "
    f"The variable that decides whether a crime series is comparable across years is "
    f"presented as chart furniture."
))
findings.append((
    "F3",
    f"The published rate is normalised by COVERED population, verified rather than assumed: "
    f"mean absolute reconstruction error {covered_err:.4f} against the covered-population "
    f"denominator and {total_err:.4f} against resident population, across "
    f"{len(denominator)} states. Recomputing a rate from offences and resident population "
    f"disagrees with the publisher by more than the year-on-year variation in crime."
))
findings.append((
    "F4",
    f"A coverage figure certifies the denominator, not the numerator. In {STATES[worst]}, "
    f"{int(quality.loc[worst, 'months under 200 offences'])} of "
    f"{int(quality.loc[worst, 'months 2016-2020'])} months in 2016-2020 carry fewer than 200 "
    f"offences, with a median month of "
    f"{quality.loc[worst, 'median monthly offences']:,.0f}, while mean reported population "
    f"coverage over the same window is {quality.loc[worst, 'mean coverage %']:.2f}%."
))
findings.append((
    "F5",
    f"The coverage figure does not support a correction. Across {len(moves)} states Pearson "
    f"r between the change in coverage and the change in the published rate is "
    f"{pearson:+.3f} (p={p_pearson:.3f}); after dropping {disqualified} on the section 3 "
    f"data-quality screen it is {ps:+.3f} on {len(screened)} states. Among the {len(losers)} "
    f"screened states losing more than 20 points of coverage the rate fell in {fell} and rose "
    f"in {rose} -- a tendency containing a sign reversal "
    f"({losers['rate change %'].idxmax()}, {losers['rate change %'].max():+.1f}%), measured on "
    f"states named rather than sampled."
))
findings.append((
    "F6",
    f"An upstream outage is distinguishable from an absence only if the client insists on it. "
    f"The gateway returns HTTP 503 intermittently and recovers within seconds; this client "
    f"retries {gateway.attempts} times and then raises "
    f"{type(outage).__name__ if 'outage' in dir() else 'CDEUpstreamUnavailable'} rather than "
    f"returning an empty frame, because an empty frame here reads as a place with no crime."
))

for tag, text in findings:
    print(f"[{tag}] {text}")
    print()

[F1] NIBRS participation is a variable, not a constant. Across 4 states and 3,552 agencies, the share reporting ranges from 17.0% (PA) to 100.0% (VA). The largest single adoption year in 2016-2023 was 2021, with 515 agencies joining.

[F2] The coverage series is published, and it is filed under 'tooltips -> Percent of Population Coverage' -- a key a chart library uses for hover text. The variable that decides whether a crime series is comparable across years is presented as chart furniture.

[F3] The published rate is normalised by COVERED population, verified rather than assumed: mean absolute reconstruction error 0.0024 against the covered-population denominator and 1.6534 against resident population, across 10 states. Recomputing a rate from offences and resident population disagrees with the publisher by more than the year-on-year variation in crime.

[F4] A coverage figure certifies the denominator, not the numerator. In Florida, 54 of 60 months in 2016-2020 carry fewer than 200 o

## Key Takeaways

1. **Participation is the variable.** In a voluntary-reporting system the set of places being
   measured changes between periods, for administrative reasons, and unevenly across
   geography. That is prior to any question about crime.

2. **Coverage certifies the denominator, never the numerator.** It tells you what share of
   people live in places served by a reporting agency. It does not tell you that those
   agencies filed usable counts for the months you care about, and the gap between those two
   can be total.

3. **Verify the denominator of any published rate.** Here the publisher divides by covered
   population — the right choice, made invisibly. The obvious reconstruction, using the
   resident population sitting in the same response, is wrong by more than the phenomenon.

4. **A coverage number supports refusal, not repair.** The direction of the bias depends on
   whether the departed agencies were above or below average, which one number cannot encode.
   If you need the transition years, fix the agency set and accept a smaller universe.

5. **An outage is not a zero.** When an upstream fails intermittently, a client that returns
   an empty result manufactures the strongest possible finding — no crime — out of a gateway
   error, and it passes every structural check.

6. **Read where a publisher files a variable as a claim about how they expect it to be used.**
   Coverage under `tooltips` is not a conspiracy; it is a hint that the series was built for a
   chart, and charts are read by people who do not check denominators.

## Exercises

1. **Build the fixed-agency comparison.** Take the agencies reporting in every year of the
   window and recompute the state rate on that set alone. Report how much smaller the universe
   becomes and whether the transition-year movement survives. State plainly which question
   your smaller series can answer and which it cannot.

2. **Find the sign.** For one state that lost coverage, use the agency roster to characterise
   the jurisdictions that stopped appearing. Can you say whether they were above or below the
   state average? If not, say precisely what data would be needed.

3. **Test the tooltip hypothesis.** Fetch a second offence type and check whether the coverage
   series is identical to the one here. If it is, coverage is a property of the agency and not
   of the offence — which changes what a per-offence analysis may assume.

4. **Reproduce §3 for your own state.** Compute median monthly offences by year alongside mean
   coverage. Decide, with a number, the earliest year from which you would use the series.

5. **Break the client deliberately.** Modify the retry loop to return an empty frame on
   exhaustion instead of raising, run §4, and report which of this notebook's conclusions
   change. That is the value of the distinction, measured.

## References

- FBI Criminal Justice Information Services. *Crime Data Explorer API* —
  `api.usa.gov/crime/fbi/cde`. Agency rosters with `is_nibrs` and `nibrs_start_date`;
  summarized state series with `populations.participated_population` and the coverage series
  under `tooltips`.
- FBI. *National Incident-Based Reporting System (NIBRS)* — participation is voluntary, and
  the Summary Reporting System was retired for national collection in January 2021.
- api.data.gov. *Developer manual* — one key authenticates across every participating agency.
- Handbook §1.4 (suppression with a flag), §8.2 (coverage is two numbers), §13 (this regime).

## Limitations

- **Ten states, not fifty.** The national picture would need 51 live calls against an upstream
  that fails intermittently, which is not a reproducible artifact. Every figure here describes
  the named states and nothing beyond them. The states were chosen to span the participation
  range, which means they are not a sample and no statistic here has a standard error.
- **One offence type.** Violent crime only. Whether coverage behaves identically for property
  offences is exercise 3, not a result.
- **The correlation in §4 has n = 10, on states named rather than sampled.** It is reported to
  establish that no usable coefficient is available here, not to estimate one: it carries no
  standard error and a different ten states would give a different number. §4 also applies
  §3's data-quality screen before recomputing, because a state whose monthly counts are not a
  measurement cannot contribute a meaningful rate change. The first version of this notebook
  left it in, and would have used a figure it had just finished disqualifying.
- **The §3 finding is about one state's series, generalised no further.** Other states in this
  set do not show the same pattern; that it happens at all, while coverage reads near-total,
  is the point.
- **No agency-level series is fetched.** The fixed-agency comparison that would actually
  repair the transition is set as exercise 1 rather than performed, because the endpoint for
  it was among those returning 503 during this build.
- **Not tested here:** whether `participated_population` is itself revised when an agency
  backfills, which would make the published rate for a past month move without any
  announcement.

**On vintage.** The FBI CDE reports agency participation as published at fetch time — the vintage is literally live — and coverage changes as agencies join. The FBI CDE serves current data with a moving max_data_date and no prior vintage, so the month range names a reference period rather than a vintage. This notebook therefore records `vintage_pinned: false` (handbook §5.2); the payload hash and retrieval time in its provenance record are what a later reader has to compare against.